# Milestone 4 — Build silver_weather

In this notebook, we transform `bronze_weather` into a clean and standardized Silver dataset.

## Focus areas:
- Type casting for numeric fields
- Region normalization
- Alert level standardization
- Filtering impossible weather values
- Creating day-level reporting field (`report_day`)
- Using reusable transformation logic with `df.transform()`

## This Silver dataset will be used for downstream analytics and business logic.

In [0]:
import yaml
from pyspark.sql import functions as F

config_path = "/Workspace/Repos/Mini Projects/Vattenfall-Engineering-Capstone-Project/config/project_config.yml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

catalog = config["catalog"]
raw_schema = config["schemas"]["raw"]
refined_schema = config["schemas"]["refined"]

## Read Bronze Weather Data

We load the raw weather data from the Bronze layer for transformation.

In [0]:
df_weather = spark.table(f"{catalog}.{raw_schema}.bronze_weather")

display(df_weather)

## Apply Standard Cleaning & Transformation

We standardize:
- region names
- measurement types
- alert levels
- numeric columns
- derive report_day from event_date

We use `df.transform()` for modular design.

In [0]:
def clean_weather(df):
    return (
        df
        .withColumn("region", F.upper(F.col("region")))
        .withColumn("weather_alert_level", F.upper(F.col("weather_alert_level")))
        .withColumn("temperature_c", F.col("temperature_c").cast("double"))
        .withColumn("wind_speed_kmh", F.col("wind_speed_kmh").cast("double"))
        .withColumn("precipitation_mm", F.col("precipitation_mm").cast("double"))
        .withColumn("report_day", F.to_date("event_date"))
    )

df_weather = df_weather.transform(clean_weather)

## Remove Invalid Weather Records

We filter out impossible values such as:
- extreme temperature values
- negative wind speed
- negative precipitation

In [0]:
df_weather = df_weather.filter(
    (F.col("temperature_c").between(-50, 60)) &
    (F.col("wind_speed_kmh") >= 0) &
    (F.col("precipitation_mm") >= 0)
)

## Clean Up Unnecessary Columns

We remove any system-generated or unwanted columns like `_rescued_data`.

In [0]:
df_weather = df_weather.drop(
    *[c for c in df_weather.columns if "rescued" in c.lower()]
)

In [0]:
display(df_weather)
df_weather.printSchema()
print("Row count:", df_weather.count())

In [0]:
df_weather.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{refined_schema}.silver_weather")

## Final Output Check

We verify the final Silver table after writing.

In [0]:
display(
    spark.table(f"{catalog}.{refined_schema}.silver_weather")
)